In [34]:
import os
import re
import httpx
import requests
from typing import Any
from google import genai
from dotenv import load_dotenv
from rich.console import Console
from rich.pretty import Pretty
from rich.theme import Theme

console = Console(
    force_terminal=True,
    force_jupyter=False,
    color_system="truecolor",
)
load_dotenv()

True

In [35]:
print(genai)


<module 'google.genai' from '/Users/sarveshmhadgut/Documents/AIPlayground/CrewAISandbox/.venv/lib/python3.12/site-packages/google/genai/__init__.py'>


In [36]:
def duckduckgo_search(query):
    url = "https://api.duckduckgo.com/"

    params = {"q": query, "format": "json", "no_html": 1, "skip_disambig": 1}

    try:
        res = requests.get(url, params=params, timeout=10)
        res.raise_for_status()

        data = res.json()

        if data.get("AbstractText"):
            return data["AbstractText"]

        if data.get("Answer"):
            return data["Answer"]

        if data.get("Definition"):
            return data["Definition"]

        related = data.get("RelatedTopics", [])

        for item in related:
            if "Text" in item:
                return item["Text"]

            if "Topics" in item:
                for sub in item["Topics"]:
                    if "Text" in sub:
                        return sub["Text"]

        return "No results found."

    except Exception as e:
        return f"Error: {e}"


In [37]:
console.print_json(data={"result": duckduckgo_search("Snoopy")})

{
  "result": "Snoopy is one of the central characters in the comic strip Peanuts by American cartoonist Charles M. Schulz. He also appears in all of the Peanuts films and television specials. Debuting in the strip on October 4, 1950, the original drawings of Snoopy were inspired by Spike, one of Schulz's childhood dogs. A largely anthropomorphic beagle, Snoopy is nearly the polar opposite of his owner, Charlie Brown: he is capable in multiple areas, quick-witted, imaginative, and independent. Snoopy usually spends his days sleeping flat on top of his doghouse or engaging in flights of fancy. This includes pretending to be a World War I flying ace who is the archenemy of the Red Baron, writing essays and short stories on his typewriter, and trying to romance the different neighborhood kids. Snoopy is also known for his large family, rivalry with Linus Van Pelt over the latter's blanket, friendship with Peppermint Patty, and longstanding best-friendship with the yellow bird Woodstock."


In [38]:
def evaluate_expression(expression):
    """Evaluates and returns mathematical expressions."""
    return eval(expression)

In [39]:
console.print_json(data={"result": str(evaluate_expression("18 * 5 / 3 + 12 / 0.5 - 10"))})

{
  "result": "44.0"
}


In [40]:
def fetch_todo(task_id):
    """Fetches a placeholder to-do item from a pseudo API."""
    url = f"https://jsonplaceholder.typicode.com/todos/{task_id}"
    res = httpx.get(url)

    if res.status_code == 200:
        return res.json()
    else:
        return {"error": "could not fetch"}


In [41]:
console.print_json(data=fetch_todo(18), indent=4)

{
    "userId": 1,
    "id": 18,
    "title": "dolorum est consequatur ea mollitia in culpa",
    "completed": false
}


In [42]:
class Llm:
    """Wrapper class for Gemini LLM"""

    def __init__(self, system: str) -> None:
        self.client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
        self.chat = self.client.chats.create(model="gemini-3.1-pro-preview")
        self.chat.send_message(message=system)

    def __call__(self, message: str) -> str:

        return self.chat.send_message(message=message).text


In [43]:
llm = Llm(system="You are a helpful assistant.")
llm("Hi")

'Hi there! How can I help you today?'

In [44]:
class ReActAgent:
    def __init__(self) -> None:
        self.memory = []
        self.system = """You are a helpful assistant. You can use the following tools:\n
                            - duckduckgo_search(query: str): to search Wikipedia\n
                            - evaluate_expression(expr: str): to evaluate a math expression\n
                            - fetch_todo(): to get a sample todo\n
                        When you need to use a tool, respond with: Action: <tool_name>[<input>]\n
                        Otherwise, respond normally as the assistant.\n"""

        self.llm = Llm(system=self.system)

    def __call__(self, message) -> Any:
        """Process a user message and return a response, using tools if necessary."""
        history = "\n".join(self.memory) + f"[Human]:'{message}'"
        res = self.llm(history)

        self.memory.append(f"Human: {message}")
        self.memory.append(f"AI: {res}")

        if res.startswith("Action:"):
            tool_call = re.match(r"Action:\s*(\w+)\[(.*?)\]", res)
            if not tool_call:
                return "Invalid tool call!"

            tool_name, tool_args = tool_call.groups()
            tool_res = self.invoke_tool(tool_name, tool_args.strip())

            tool_message = f"Observation: {tool_res}"
            self.memory.append(tool_message)

            return tool_message
        else:
            return res

    def invoke_tool(self, tool_name, tool_args):
        """Invoke execution of an agent tool."""
        try:
            if tool_name == "duckduckgo_search":
                return duckduckgo_search(tool_args)
            elif tool_name == "evaluate_expression":
                return evaluate_expression(tool_args)
            elif tool_name == "fetch_todo":
                return fetch_todo(tool_args)
            else:
                return f"Unknown tool: {tool_name}"
        except Exception as e:
            return f"Error executing {tool_name}: {e}"

In [45]:
agent = ReActAgent()
print(agent("Hello!"))

Hello! How can I help you today?


In [46]:
print(agent("Is Strait of Hormuz Open?"))

Observation: The Strait of Hormuz is a waterway between the Persian Gulf and the Gulf of Oman. On the north coast lies Iran, and on the south coast lies the Musandam Peninsula under the Musandam Governorate of Oman, with a portion of the southwest of the peninsula under the United Arab Emirates. The strait is about 104 miles long, with a width varying from about 60 mi to 24 mi. It provides the only sea passage from the Persian Gulf to the open ocean and is one of the world's most strategically important choke points. During 2023–2025, 20% of the world's liquefied natural gas and 25% of seaborne oil trade passed through the strait annually. It is a major source of petroleum products for Europe and Asia and has been described as "critical" to Europe's energy security. It is also the only maritime route for several Gulf countries including UAE, Qatar, Bahrain, Kuwait and Iraq and disruption to the strait can cause severe supply shortages.


In [47]:
print(agent("What is 18**5 / 3**5 + 12 / 0.5 - 10?"))

Observation: 7790.0
